# Joint Control

Select one robot, Dex3, or Inspire hand joint and move it with bounded pose steps.

In [ ]:
from pathlib import Path
import sys

MODULE_DIR = Path.cwd()
if MODULE_DIR.name == "scripts":
    MODULE_DIR = MODULE_DIR.parent
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from sdk_client import Robot

rob = Robot("eth0")

In [ ]:
import math
import time

import ipywidgets as widgets
from IPython.display import display

from inspire_sdk import (
    ANGLE_SET_REGISTER,
    CLEAR_ERROR_REGISTER,
    FORCE_SET_REGISTER,
    HAND_CONFIGS,
    SPEED_SET_REGISTER,
    ModbusTcp,
)
from sdk_client import BODY_JOINT_NAME_BY_INDEX, UPPER_BODY_JOINTS, WAIST_HOLD_KD, WAIST_HOLD_KP
from sdk_hand import HAND_MAX_LIMITS, HAND_MIN_LIMITS, hand_open_targets

MAX_STEP_RAD = 0.1
ROBOT_RATE_HZ = 50.0
DEX3_RATE_HZ = 50.0
INSPIRE_SPEED = 120
INSPIRE_FORCE = 120
INSPIRE_MAX_RAD = 1.57
INSPIRE_UNITS_PER_RAD = 1000.0 / INSPIRE_MAX_RAD

ROBOT_JOINT_LIMITS = {
    12: (-2.618, 2.618),
    13: (-0.52, 0.52),
    14: (-0.52, 0.52),
    15: (-3.0892, 2.6704),
    16: (-1.5882, 2.2515),
    17: (-2.618, 2.618),
    18: (-1.0472, 2.0944),
    19: (-1.9722, 1.9722),
    20: (-1.6144, 1.6144),
    21: (-1.6144, 1.6144),
    22: (-3.0892, 2.6704),
    23: (-2.2515, 1.5882),
    24: (-2.618, 2.618),
    25: (-1.0472, 2.0944),
    26: (-1.9722, 1.9722),
    27: (-1.6144, 1.6144),
    28: (-1.6144, 1.6144),
}

DEX3_JOINT_NAMES = ["thumb_0", "thumb_1", "thumb_2", "middle_0", "middle_1", "index_0", "index_1"]
INSPIRE_JOINT_NAMES = ["joint_0", "joint_1", "joint_2", "joint_3", "thumb_0", "thumb_1"]

def _clamp(value, lo, hi):
    return max(float(lo), min(float(hi), float(value)))


def _bounded_path(start, target, step=MAX_STEP_RAD):
    start = float(start)
    target = float(target)
    delta = target - start
    steps = max(1, int(math.ceil(abs(delta) / max(1e-6, float(step)))))
    return [start + delta * (idx / steps) for idx in range(1, steps + 1)]


joint_specs = []
for joint in UPPER_BODY_JOINTS:
    lo, hi = ROBOT_JOINT_LIMITS[int(joint)]
    joint_specs.append({
        "kind": "robot",
        "label": f"Robot {BODY_JOINT_NAME_BY_INDEX[int(joint)]} ({joint})",
        "joint": int(joint),
        "min": lo,
        "max": hi,
        "units": "rad",
    })
for hand in ("left", "right"):
    for idx, name in enumerate(DEX3_JOINT_NAMES):
        joint_specs.append({
            "kind": "dex3",
            "label": f"Dex3 {hand} {name}",
            "hand": hand,
            "idx": idx,
            "min": HAND_MIN_LIMITS[hand][idx],
            "max": HAND_MAX_LIMITS[hand][idx],
            "units": "rad",
        })
for hand in ("left", "right"):
    for idx, name in enumerate(INSPIRE_JOINT_NAMES):
        joint_specs.append({
            "kind": "inspire",
            "label": f"Inspire {hand} {name}",
            "hand": hand,
            "idx": idx,
            "min": 0.0,
            "max": INSPIRE_MAX_RAD,
            "units": "rad mapped to 0..1000 register",
        })

spec_by_key = {f"{spec['kind']}:{spec.get('hand', '')}:{spec.get('joint', spec.get('idx'))}": spec for spec in joint_specs}
options = [(spec["label"], key) for key, spec in spec_by_key.items()]
target_cache = {}
dex3_targets = {hand: hand_open_targets(hand) for hand in ("left", "right")}

joint_dropdown = widgets.Dropdown(
    options=options,
    value=options[0][1],
    description="Joint",
    layout=widgets.Layout(width="520px"),
)
pose_slider = widgets.FloatSlider(
    description="Pose",
    min=joint_specs[0]["min"],
    max=joint_specs[0]["max"],
    step=0.01,
    value=0.0,
    continuous_update=False,
    readout_format=".3f",
    layout=widgets.Layout(width="650px"),
)
refresh_button = widgets.Button(description="Read current")
zero_hand_button = widgets.Button(description="Zero selected hand")
status = widgets.Output()
_updating = False


def _current_robot_value(spec):
    value = rob.get_joint_position(int(spec["joint"]))
    if value is None:
        value = 0.0
    return _clamp(value, spec["min"], spec["max"])


def _current_dex3_value(spec):
    hand = spec["hand"]
    idx = int(spec["idx"])
    controller = rob._get_hand(hand)
    snapshot = controller.get_state_snapshot(max_age=1.0)
    if snapshot is not None:
        dex3_targets[hand] = list(snapshot["positions"])
    return _clamp(dex3_targets[hand][idx], spec["min"], spec["max"])


def _current_inspire_value(spec, key):
    return float(target_cache.get(key, 0.0))


def _current_value(key):
    spec = spec_by_key[key]
    if spec["kind"] == "robot":
        return _current_robot_value(spec)
    if spec["kind"] == "dex3":
        return _current_dex3_value(spec)
    return _current_inspire_value(spec, key)


def _configure_slider(key, read_current=True):
    global _updating
    spec = spec_by_key[key]
    value = _current_value(key) if read_current else float(target_cache.get(key, spec["min"]))
    value = _clamp(value, spec["min"], spec["max"])
    target_cache[key] = value
    _updating = True
    try:
        pose_slider.min = float(spec["min"])
        pose_slider.max = float(spec["max"])
        pose_slider.value = value
    finally:
        _updating = False
    with status:
        status.clear_output()
        print(f"{spec['label']}: {value:.3f} {spec['units']}")


def _command_robot(spec, value):
    rob.move_upper_body_joint(
        int(spec["joint"]),
        float(value),
        command_rate_hz=ROBOT_RATE_HZ,
        max_speed_rad_s=0.45,
        kp=30.0,
        kd=1.5,
        waist_kp=WAIST_HOLD_KP,
        waist_kd=WAIST_HOLD_KD,
        timeout=3.0,
    )


def _command_dex3(spec, value):
    hand = spec["hand"]
    idx = int(spec["idx"])
    targets = list(dex3_targets[hand])
    targets[idx] = _clamp(value, spec["min"], spec["max"])
    dex3_targets[hand] = targets
    rob.hand_pose(targets, hand=hand, hold_s=0.12, rate_hz=DEX3_RATE_HZ, kp=0.8, kd=0.05, tau=0.0, ramp_s=0.08)


def _command_inspire(spec, value):
    hand = spec["hand"]
    idx = int(spec["idx"])
    register_value = int(round(_clamp(value, 0.0, INSPIRE_MAX_RAD) * INSPIRE_UNITS_PER_RAD))
    register_value = max(0, min(1000, register_value))
    config = HAND_CONFIGS[hand]
    with ModbusTcp(config.ip, config.port, config.unit_id) as client:
        client.write_single_register(CLEAR_ERROR_REGISTER, 1)
        client.write_single_register(SPEED_SET_REGISTER + idx, INSPIRE_SPEED)
        client.write_single_register(FORCE_SET_REGISTER + idx, INSPIRE_FORCE)
        client.write_single_register(ANGLE_SET_REGISTER + idx, register_value)


def _send_pose(key, target):
    spec = spec_by_key[key]
    start = float(target_cache.get(key, _current_value(key)))
    target = _clamp(target, spec["min"], spec["max"])
    with status:
        status.clear_output()
        print(f"Moving {spec['label']}: {start:.3f} -> {target:.3f}; max command step {MAX_STEP_RAD:.3f} rad")
    for value in _bounded_path(start, target):
        if spec["kind"] == "robot":
            _command_robot(spec, value)
        elif spec["kind"] == "dex3":
            _command_dex3(spec, value)
        else:
            _command_inspire(spec, value)
            time.sleep(0.05)
        target_cache[key] = float(value)
    with status:
        status.clear_output()
        print(f"Done: {spec['label']} = {target:.3f} {spec['units']}")


def _on_joint_change(change):
    if change["name"] == "value":
        _configure_slider(change["new"])


def _on_pose_change(change):
    if _updating or change["name"] != "value":
        return
    _send_pose(joint_dropdown.value, change["new"])


def _on_refresh(_button):
    _configure_slider(joint_dropdown.value)


def _on_zero_selected_hand(_button):
    key = joint_dropdown.value
    spec = spec_by_key[key]
    if spec["kind"] == "dex3":
        rob.zero_torque_fingers(spec["hand"], persistent=True)
        message = f"Dex3 {spec['hand']} fingers released."
    elif spec["kind"] == "inspire":
        from inspire_sdk import zero_torque_hand
        zero_torque_hand(spec["hand"])
        message = f"Inspire {spec['hand']} hand released."
    else:
        message = "Selected joint is a robot body joint; hand zero torque was not sent."
    with status:
        status.clear_output()
        print(message)


joint_dropdown.observe(_on_joint_change, names="value")
pose_slider.observe(_on_pose_change, names="value")
refresh_button.on_click(_on_refresh)
zero_hand_button.on_click(_on_zero_selected_hand)

_configure_slider(joint_dropdown.value)
display(widgets.VBox([joint_dropdown, pose_slider, widgets.HBox([refresh_button, zero_hand_button]), status]))